# AdeSEG observable smoke stage

This is a non-claim-bearing preflight for the preregistered utility-memory experiment. It proves that Kaggle can clone the pinned branch, reach the configured W&B Forge project, resolve the attached datasets, acquire the fixed SAM2.1 Hiera-S checkpoint, build the instrumented predictor, and analyze one utility frame. Every boundary writes a status file and logs to W&B.

In [ ]:
import importlib.util, json, os, shutil, subprocess, sys, time
from pathlib import Path
from kaggle_secrets import UserSecretsClient

WORK = Path('/kaggle/working')
REPO = WORK / 'AdeSEG'
OUT = WORK / 'outputs' / '16:10:2026' / 'stage0_smoke'
OUT.mkdir(parents=True, exist_ok=True)
os.environ.update({
    'WANDB_API_KEY': UserSecretsClient().get_secret('WANDB_API_KEY'),
    'WANDB_BASE_URL': 'https://forge.coreweave.com/api/wandb',
    'WANDB_PROJECT': 'ade-seg',
    'WANDB_ENTITY': 'adenoid-hypertrophy',
    'PYTHONUNBUFFERED': '1',
    'GIT_TERMINAL_PROMPT': '0',
})

if importlib.util.find_spec('wandb') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'wandb'], check=True, timeout=300)
import wandb
monitor = wandb.init(project='ade-seg', entity='adenoid-hypertrophy', name='kaggle-stage0-observable-smoke-2026-10-16', tags=['utility', 'kaggle', 'smoke', 'non-claim-bearing'], settings=wandb.Settings(init_timeout=60))
started = time.time()

def mark(stage, state='running', **details):
    payload = {'stage': stage, 'state': state, 'elapsed_seconds': time.time() - started, **details}
    temporary = OUT / 'status.json.tmp'
    temporary.write_text(json.dumps(payload, indent=2) + '\n')
    os.replace(temporary, OUT / 'status.json')
    monitor.log({'status/elapsed_seconds': payload['elapsed_seconds'], 'status/stage': stage, 'status/state': state})
    print('[status]', json.dumps(payload, sort_keys=True), flush=True)

def run(command, cwd=None, timeout=900):
    command = list(map(str, command))
    mark('command', command=' '.join(command))
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, env=os.environ.copy(), check=True, timeout=timeout)

mark('wandb_connected')


In [ ]:
mark('repository_setup')
if REPO.exists():
    shutil.rmtree(REPO)
run(['git', 'clone', '--depth', '1', '--branch', 'experiment/utility-aware-memory-2026-10-16', 'https://github.com/aCoderChild/AdeSEG.git', REPO], timeout=300)
run(['git', 'submodule', 'update', '--init', '--depth', '1', 'external/MedSAM2'], cwd=REPO, timeout=600)
torch_before = __import__('torch').__version__
missing = [package for module, package in {'hydra': 'hydra-core>=1.3.2', 'iopath': 'iopath>=0.1.10'}.items() if importlib.util.find_spec(module) is None]
if missing:
    run([sys.executable, '-m', 'pip', 'install', *missing], timeout=300)
if importlib.util.find_spec('tensordict') is None:
    run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'tensordict==0.6.2'], timeout=300)
import torch
assert torch.__version__ == torch_before and torch.cuda.is_available(), (torch_before, torch.__version__, torch.cuda.is_available())
mark('repository_ready', torch=torch.__version__, gpu=torch.cuda.get_device_name(0))


In [ ]:
mark('data_and_checkpoint')
PROTOCOL = REPO / 'protocols/utility_retrieval_v1.json'
DATA = WORK / 'experiment_data'
DATA.mkdir(exist_ok=True)
DAVIS_MANIFEST = DATA / 'davis.jsonl'
run([sys.executable, REPO/'scripts/build_davis_manifest.py', '--data-root', '/kaggle/input', '--protocol', PROTOCOL, '--output', DAVIS_MANIFEST], cwd=REPO, timeout=120)
CKPT = DATA / 'sam2.1_hiera_small.pt'
if not CKPT.exists():
    run(['wget', '--show-progress', '--timeout=30', '--tries=3', 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt', '-O', CKPT], timeout=300)
assert CKPT.stat().st_size > 100_000_000, CKPT.stat().st_size
mark('checkpoint_ready', checkpoint_bytes=CKPT.stat().st_size)


In [ ]:
mark('instrumented_utility_smoke')
SMOKE = OUT / 'utility_bear_one_frame'
run([sys.executable, REPO/'scripts/generate_utility_labels.py', '--sam2-cfg', 'configs/sam2.1_hiera_s1024.yaml', '--sam2-checkpoint', CKPT, '--manifest', DAVIS_MANIFEST, '--split', 'val', '--video-ids', 'bear', '--label-ids', '1', '--protocol', PROTOCOL, '--output-dir', SMOKE, '--device', 'cuda', '--max-analyzed-frames', '1', '--frame-selection', 'every_n', '--every-n', '20', '--wandb-base-url', os.environ['WANDB_BASE_URL'], '--wandb-name', 'stage0-bear-one-frame'], cwd=REPO, timeout=1800)
summary = json.loads((SMOKE/'summary.json').read_text())
status = json.loads((SMOKE/'status.json').read_text())
assert summary['analyzed_frames'] == 1, summary
assert status['state'] == 'complete', status
mark('complete', state='complete', analyzed_frames=summary['analyzed_frames'], no_op_max_abs=summary['sanity_no_op_max_abs'])
artifact = wandb.Artifact('ade-seg-stage0-smoke-2026-10-16', type='experiment-preflight')
for path in OUT.rglob('*.json'):
    artifact.add_file(str(path), name=str(path.relative_to(OUT)))
monitor.log_artifact(artifact)
monitor.finish()
print('SMOKE COMPLETE:', json.dumps(summary, indent=2), flush=True)
